<a href="https://colab.research.google.com/github/Kubojah-Dan/kuboja-codeboosters-2026/blob/main/DAY3/Day3_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install requests --quiet
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print(f"Pandas : {pd.__version__}")
print(f"Requests : {requests.__version__}")


All libraries imported successfully
Pandas : 2.2.2
Requests : 2.32.4


In [23]:
raw_df = pd.read_csv("/content/messy_sales_data.csv")
print(f"Raw Data loaded:{raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
print(raw_df.columns.tolist())
print("First 5 rows without index:")
print(raw_df.head(5).to_string(index=False))

Raw Data loaded:30 rows, 9 columns
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']
First 5 rows without index:
 order_id customer_name  product    category  quantity  unit_price order_date      city   sales_rep
     1001  Ramesh Kumar   Laptop Electronics       2.0       45000 2024-01-05    Mumbai Anil Sharma
     1002    Priya Nair      NaN Electronics       1.0       15000 2024-01-07     Delhi  Sunita Rao
     1003    AMIT VERMA Keyboard Accessories       3.0        1200 2024-01-08 Bangalore Anil Sharma
     1004  Sunita Patel  Monitor Electronics       NaN       22000 2024-01-10   Chennai  Ravi Kumar
     1005  Ramesh Kumar   Laptop Electronics       2.0       45000 2024-01-05    Mumbai Anil Sharma


In [24]:
#==========================================
# Adding Duplicates
#==========================================

new_row = pd.DataFrame({
    'order_id' : [1031],
    'customer_name' : ['Ramesh Kumar'],
    'product' : ['Laptop'],
    'category' : ['Electronics'],
    'quantity' : [2.0],
    'unit_price' : [45000],
    'order_date' : ['2024-01-05'],
    'city' : ['Mumbai'],
    'sales_rep' : ['Anil Sharma']
})
raw_df = pd.concat([raw_df, new_row], ignore_index=True)

In [26]:
print("="*55)
print("            DATA QUALITY DIAGNOSIS REPORT")
print("="*55)

print("1. Missing Values per column")
print(raw_df.isnull().sum())

print("2. Duplicate rows")
print(raw_df[raw_df.duplicated()])

print("3. Data types")
print(raw_df.dtypes)

print("4. Unique  Categories", raw_df['category'].unique())
print("5. ")

            DATA QUALITY DIAGNOSIS REPORT
1. Missing Values per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
2. Duplicate rows
Empty DataFrame
Columns: [order_id, customer_name, product, category, quantity, unit_price, order_date, city, sales_rep]
Index: []
3. Data types
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
4. Unique  Categories ['Electronics' 'Accessories' nan]
5. 


In [27]:
print(f"\n[2] Duplicate Rows: {raw_df['quantity'].duplicated().isnull().sum()}")


[2] Duplicate Rows: 0


In [28]:
df = raw_df.copy()
print(df)

    order_id  customer_name     product     category  quantity  unit_price  \
0       1001   Ramesh Kumar      Laptop  Electronics       2.0       45000   
1       1002     Priya Nair         NaN  Electronics       1.0       15000   
2       1003     AMIT VERMA    Keyboard  Accessories       3.0        1200   
3       1004   Sunita Patel     Monitor  Electronics       NaN       22000   
4       1005   Ramesh Kumar      Laptop  Electronics       2.0       45000   
5       1006    kiran mehta       Mouse  Accessories      10.0         800   
6       1007   Deepak Singh  Headphones  Electronics       2.0        3500   
7       1008            NaN      Webcam  Accessories       1.0        2500   
8       1009     Ananya Das      Laptop  Electronics       1.0       45000   
9       1010    Vikram Iyer    Keyboard  Accessories       5.0        1200   
10      1011    Pooja Gupta     Monitor  Electronics       2.0       22000   
11      1012     SURESH RAO     USB Hub  Accessories       8.0  

In [29]:
#=============================================
# Handle Missing Values
#=============================================

print("Before fixing nulls:", df.isnull().sum().sum(), 'total missing values')
df['customer_name'].fillna('Unknown Customer', inplace=True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty, inplace=True)
print(f" Filled missing quantity with median: {median_qty}")
df['category'].fillna('Uncategorized', inplace=True)
print('After fixing nulls: ', df.isnull().sum().sum(), 'total missing values')

Before fixing nulls: 7 total missing values
 Filled missing quantity with median: 2.0
After fixing nulls:  1 total missing values


In [32]:
#==========================================
# Removing Duplicates
#==========================================

print(f"Before duplication: {len(df)} rows")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("\nDuplicate rows:")
print(df[df.duplicated(keep=False)][['customer_name', 'product', 'order_date', 'city', 'sales_rep', 'category', 'unit_price']].head())

df.drop_duplicates(inplace=True)

print(f"\nAfter duplication: {len(df)} rows")
print(f"Rows removed: {len(raw_df) - len(df)}")

Before duplication: 31 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [customer_name, product, order_date, city, sales_rep, category, unit_price]
Index: []

After duplication: 31 rows
Rows removed: 0


In [41]:
print("Sample dates before parsing: ")
print(df['order_date'].head(5).tolist())

df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)
nat_count = df['order_date'].isnull().sum()
print(f"\nUnparseable dates (NaT): {nat_count}")

df['day'] = df['order_date'].dt.strftime('%d')
df['month'] = df['order_date'].dt.strftime('%m')
df['year'] = df['order_date'].dt.strftime('%Y')
df['month_name'] = df['order_date'].dt.strftime('%B')

print('\nSample Date after parsing:')
print(df[['order_date', 'day', 'month_name', 'year']].head(5))

Sample dates before parsing: 
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00')]

Unparseable dates (NaT): 2

Sample Date after parsing:
  order_date day month_name  year
0 2024-01-05  05    January  2024
1 2024-01-07  07    January  2024
2 2024-01-08  08    January  2024
3 2024-01-10  10    January  2024
4 2024-01-05  05    January  2024
